In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


#Config class
class GPTConfig:
    vocab_size = 20000
    block_size = 256
    n_layers = 3
    n_heads = 4
    d_model = 128
    d_ff = 512
    dropout = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.d_model % config.n_heads == 0

        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.proj = nn.Linear(config.d_model, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size)
        )

    def forward(self, x):
        B, T, C = x.size()

        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.proj(y)
        return y


#Transformer
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff),
            nn.GELU(),
            nn.Linear(config.d_ff, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x



class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.block_size, config.d_model)

        self.blocks = nn.Sequential(*[Block(config) for _ in range(config.n_layers)])
        self.ln_f = nn.LayerNorm(config.d_model)

        self.head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.head.weight = self.token_embedding.weight

        self.block_size = config.block_size

    def forward(self, idx, targets=None):
        B, T = idx.size()

        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)

        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(pos)

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx

In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text()
    return full_text


In [ ]:
import tiktoken
from torch.utils.data import Dataset, DataLoader

# Tokenize the book
enc = tiktoken.get_encoding("gpt2")

pdf_file_path = '<pdf file path>'
text = extract_text_from_pdf(pdf_file_path)

tokens = enc.encode(text)

class BookDataset(Dataset):
    def __init__(self, tokens, block_size):
        self.tokens = torch.tensor(tokens, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return len(self.tokens) - self.block_size

    def __getitem__(self, i):
        x = self.tokens[i : i + self.block_size]
        y = self.tokens[i + 1 : i + self.block_size + 1]
        return x, y

In [ ]:
config = GPTConfig()
config.vocab_size = 50257  # match tiktoken

model = GPT(config)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

dataset = BookDataset(tokens, config.block_size)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for epoch in range(10):
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, targets=y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 3.8645
Epoch 1 | Loss: 2.9500
Epoch 2 | Loss: 2.2901
Epoch 3 | Loss: 1.8876
Epoch 4 | Loss: 1.3859
Epoch 5 | Loss: 1.7053
Epoch 6 | Loss: 1.2709
Epoch 7 | Loss: 0.9328
Epoch 8 | Loss: 1.0424
Epoch 9 | Loss: 0.7777


In [ ]:
prompt = "<prompt about the content>"
tokens_in = torch.tensor(enc.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
output = model.generate(tokens_in, max_new_tokens=200)
print(enc.decode(output[0].tolist()))

Deep learning is the fully conveyororor er e of Keras
Tuner, Dan capable of dense projections enables deep learning, complex representations are done by primarily
there are all other words in the same way to break break break301Advanced use-
n’s necessary to natural f r a continuous u r e c i n i n t i o n ,  G u l e -  p r o o d a t s  n e r o n i n g  t h a t  a n d  e
optimization.
stacking to process is done via different of image that explore explore.
 Depending on K-to benchmark: the probability distribution of any kernel—the workshop, it’s notice these applications. But these interest are
machine learning, the generator is partic-performing characteristics of
specificspecific information through
250 of real images and decoded as points.
Before proceed: the smallest from the image. So this model outputs are
